In [5]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="https://huggingface.co/Shankar009/konkani-tokenizer_lts/resolve/main/spm_konkani.model",
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<s>",
    eos_token="</s>"
)
print(tokenizer.tokenize("काँय जाले रे"))


Exception: The filename, directory name, or volume label syntax is incorrect. (os error 123)

In [8]:
from transformers import PreTrainedTokenizer, PreTrainedTokenizerFast
import sentencepiece as spm

# Step 1: Load the SentencePiece model
sp = spm.SentencePieceProcessor()
sp.load("spm_konkani.model")

# Step 2: Create slow tokenizer
class KonkaniTokenizer(PreTrainedTokenizer):
    def __init__(self, sp_model):
        self.sp_model = sp_model
        super().__init__(
            unk_token="<unk>",
            pad_token="<pad>",
            bos_token="<s>",
            eos_token="</s>",
        )

    def _tokenize(self, text):
        return self.sp_model.encode(text, out_type=str)

    def _convert_token_to_id(self, token):
        return self.sp_model.piece_to_id(token)

    def _convert_id_to_token(self, index):
        return self.sp_model.id_to_piece(index)

    def convert_tokens_to_string(self, tokens):
        return self.sp_model.decode(tokens)

    def get_vocab(self):
        return {self.sp_model.id_to_piece(i): i for i in range(self.sp_model.get_piece_size())}

# Step 3: Wrap slow tokenizer in fast tokenizer
slow_tokenizer = KonkaniTokenizer(sp)
fast_tokenizer = PreTrainedTokenizerFast(tokenizer_object=slow_tokenizer)

# Save in HF format
fast_tokenizer.save_pretrained("konkani-tokenizer_hf")


AttributeError: KonkaniTokenizer has no attribute truncation